### OPW Episode Cohort Analysis

Builds its own 6 month Orange / Purple / WhatsApp cohort from Athena.
Does not use FinalData.xlsx.

Output: `OPW_Episode_Cohort_Analysis.xlsx` (16 sheets). Cells after the export step print the same tables in the notebook.


In [18]:
import json
import os
import time
from datetime import datetime
from pathlib import Path

import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

ANALYSIS_MONTHS = 6
ANALYSIS_END = pd.Timestamp.now().normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
WINDOW_START = ANALYSIS_END - pd.DateOffset(months=ANALYSIS_MONTHS)
WINDOW_START_STR = WINDOW_START.strftime("%Y-%m-%d %H:%M:%S")

load_dotenv(Path.cwd() / ".env")


def _env(name, default=""):
    return (os.getenv(name) or default).strip()


AWS_REGION = _env("AWS_REGION", "ap-south-1")
ATHENA_DB = _env("ATHENA_DATABASE", "leadsdb-prod")
ATHENA_WORKGROUP = _env("ATHENA_WORKGROUP", "asbl_analytics_prod")
S3_OUTPUT = _env("ATHENA_OUTPUT_LOCATION", "s3://airflow-etl-db/athena_storage/")
AWS_ACCESS_KEY_ID = _env("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = _env("AWS_SECRET_ACCESS_KEY")

# ponytail: this laptop has no ASBL profile; .env keys are the working auth path
session_kwargs = {"region_name": AWS_REGION}
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    session_kwargs.update(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
elif "ASBL" in boto3.Session().available_profiles:
    session_kwargs["profile_name"] = "ASBL"

session = boto3.Session(**session_kwargs)


def athena_read(sql):
    return wr.athena.read_sql_query(
        sql=sql,
        database=ATHENA_DB,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=S3_OUTPUT,
        ctas_approach=False,
    )


def athena_batches(sql_template, ids, batch_size=500):
    clean = list(
        {
            str(x).replace(".0", "").strip()
            for x in ids
            if pd.notna(x) and str(x).strip() not in ("", "nan", "None")
        }
    )
    parts = [athena_read(sql_template.format(ids=", ".join(f"'{b}'" for b in clean[i : i + batch_size])))
             for i in range(0, len(clean), batch_size)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


_leadSourceDF = athena_read(f'SELECT _id, name FROM "{ATHENA_DB}"."leadsources"')
_projectDF = athena_read(f'SELECT _id, name FROM "{ATHENA_DB}"."projects"')


def id_name_map(lookup_df):
    return dict(zip(
        lookup_df["_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip(),
        lookup_df["name"].astype(str).str.strip(),
    ))


SOURCE_NAME_MAP = id_name_map(_leadSourceDF)
PROJECT_NAME_MAP = id_name_map(_projectDF)


def map_id_col(df, col, lookup):
    if col not in df.columns:
        return df
    ids = df[col].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    blank = df[col].isna() | ids.isin(["", "nan", "None", "<NA>", "NaT"])
    mapped = ids.map(lookup)
    df[col] = mapped.where(mapped.notna() & ~blank, ids.where(~blank, pd.NA))
    return df


def replace_ids_with_names(df):
    df = map_id_col(df, "project", PROJECT_NAME_MAP)
    df = map_id_col(df, "bornSource", SOURCE_NAME_MAP)
    df = map_id_col(df, "source", SOURCE_NAME_MAP)
    df = map_id_col(df, "episode_source", SOURCE_NAME_MAP)
    df.drop(
        columns=["episode_source_id", "episode_source_name", "project_name", "bornSource_name", "source_name"],
        errors="ignore",
        inplace=True,
    )
    return df


print(f"Athena: db={ATHENA_DB} workgroup={ATHENA_WORKGROUP}")
print(f"Analysis window: {WINDOW_START.date()} to {ANALYSIS_END.date()}")
print(f"Lead sources loaded: {len(_leadSourceDF):,}")
print(f"Projects loaded: {len(_projectDF):,}")


Athena: db=leadsdb-prod workgroup=asbl_analytics_prod
Analysis window: 2026-02-19 to 2026-08-19
Lead sources loaded: 315
Projects loaded: 12


In [19]:
import json
from datetime import datetime
from pathlib import Path

import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from scipy import stats

EPISODE_ORDER = ["Episode 1", "Episode 2", "Episode 3", "Episode 4", "Episode 5+"]
META_EXCLUDE = {
    "lead", "lead_clean", "leadId", "lead_oid", "masterLeadId", "globalId", "visitorId",
    "project", "status", "episode_id", "episode_number", "episode_group",
    "episode_start_time", "episode_end_time", "lead_createdAt", "start_stage", "end_stage",
    "stage_before_ni", "stage_transition_path", "cancellation_reason_at_episode_end",
    "bornSource", "source", "secondarySource", "episode_source",
    "original_source", "qualifying_source", "reactivation_source", "resubmission_source",
    "qualifying_event_types", "qualifying_channels", "is_orange_lead", "is_purple_lead",
    "is_whatsapp_lead", "first_qualifying_event_time", "last_qualifying_event_time",
    "qualifying_event_count", "prev_episode_number", "lead_curr", "lead_prev", "transition_pair",
}

def _pick_col(df, *candidates):
    lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in df.columns:
            return name
        if name.lower() in lower:
            return lower[name.lower()]
    return None

def _to_bool(series):
    return series.astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes"])

def _parse_secondary_events(val):
    if not isinstance(val, str) or not val.strip() or val.strip() == "[]":
        return []
    try:
        parsed = json.loads(val)
    except Exception:
        return []
    return [x for x in parsed if isinstance(x, dict)] if isinstance(parsed, list) else []

def _classify_event_type(event_time, lead_created, episodes):
    if pd.isna(event_time):
        return "unknown"
    if pd.notna(lead_created) and abs((event_time - lead_created).total_seconds()) <= 86400:
        return "born"
    if episodes is not None and not episodes.empty:
        ep_starts = pd.to_datetime(episodes["episode_start_time"], errors="coerce")
        if ((ep_starts - event_time).abs() <= pd.Timedelta(days=1)).any():
            ep_match = episodes.loc[(ep_starts - event_time).abs() <= pd.Timedelta(days=1)]
            if (ep_match["episode_number"] > 1).any():
                return "reactivated"
    return "resubmitted"

def resolve_lead_flag_columns(leads_df):
    return {
        "is_orange": _pick_col(leads_df, "is_orange", "isorange", "isOrange"),
        "is_purple": _pick_col(leads_df, "is_purple", "ispurple", "ispurplelead", "isPurple"),
        "orange_time": _pick_col(leads_df, "orange_lead_time", "orangeleadtime", "orangeLeadTime", "is_orange_time"),
        "purple_time": _pick_col(leads_df, "purple_lead_time", "purpleleadtime", "purpleLeadTime", "is_purple_time"),
        "born_source": _pick_col(leads_df, "bornSource", "bornsource"),
        "secondary_source": _pick_col(leads_df, "secondarySource", "secondarysource"),
        "lead_created": _pick_col(leads_df, "lead_createdAt", "createdat", "createdAt"),
        "lead_id": _pick_col(leads_df, "lead", "_id"),
    }

def build_whatsapp_source_ids(source_df):
    name_col = _pick_col(source_df, "name")
    id_col = _pick_col(source_df, "_id")
    if not name_col or not id_col:
        return set()
    mask = source_df[name_col].astype(str).str.contains("whatsapp", case=False, na=False)
    return set(source_df.loc[mask, id_col].astype(str).str.strip())

def extract_lead_qualifying_events(lead_row, cols, wa_source_ids, source_name_map, window_start, window_end, episodes=None):
    """Return qualifying events for one lead.

    Cohort logic is OR across channels:
      - Orange timestamp in window
      - Purple timestamp in window
      - WhatsApp born / reactivated / resubmitted in window

    Orange and Purple do NOT require a specific bornSource.
    WhatsApp reactivation/resubmission also does NOT require WhatsApp birth.
    """
    events = []
    lead_created = pd.to_datetime(lead_row.get(cols["lead_created"]), errors="coerce") if cols["lead_created"] else pd.NaT

    def add_event(channel, event_time, event_type=None):
        if pd.isna(event_time) or event_time < window_start or event_time > window_end:
            return
        events.append({
            "channel": channel,
            "event_time": event_time,
            "event_type": event_type or _classify_event_type(event_time, lead_created, episodes),
        })

    # --- Orange (any original birth source) ---
    if cols["orange_time"]:
        orange_ts = pd.to_datetime(lead_row.get(cols["orange_time"]), errors="coerce")
        if pd.notna(orange_ts):
            add_event("Orange", orange_ts)

    # --- Purple (any original birth source) ---
    if cols["purple_time"]:
        purple_ts = pd.to_datetime(lead_row.get(cols["purple_time"]), errors="coerce")
        if pd.notna(purple_ts):
            add_event("Purple", purple_ts)

    # --- WhatsApp born ---
    born_source = str(lead_row.get(cols["born_source"], "")).strip() if cols["born_source"] else ""
    born_source_name = source_name_map.get(born_source, born_source).lower()
    if born_source in wa_source_ids or "whatsapp" in born_source_name:
        add_event("WhatsApp", lead_created, "born")

    # --- WhatsApp / Orange / Purple reactivation or resubmission from secondarySource ---
    for item in _parse_secondary_events(lead_row.get(cols["secondary_source"]) if cols["secondary_source"] else None):
        event_time = pd.to_datetime(item.get("date"), errors="coerce")
        if pd.isna(event_time) or event_time < window_start or event_time > window_end:
            continue

        src = str(item.get("source", "")).strip()
        src_name = source_name_map.get(src, src).lower()
        is_react = item.get("isReactivation")
        is_reactivation = is_react is True or str(is_react).lower() == "true"
        event_type = "reactivated" if is_reactivation else "resubmitted"

        # WhatsApp reactivation/resubmission (born by any source)
        if src in wa_source_ids or "whatsapp" in src_name:
            add_event("WhatsApp", event_time, event_type)

        # Orange/Purple event aligned to this secondarySource event timestamp
        if cols["orange_time"]:
            orange_ts = pd.to_datetime(lead_row.get(cols["orange_time"]), errors="coerce")
            if pd.notna(orange_ts) and abs((orange_ts - event_time).total_seconds()) <= 86400:
                add_event("Orange", event_time, event_type)
        if cols["purple_time"]:
            purple_ts = pd.to_datetime(lead_row.get(cols["purple_time"]), errors="coerce")
            if pd.notna(purple_ts) and abs((purple_ts - event_time).total_seconds()) <= 86400:
                add_event("Purple", event_time, event_type)

    return events

def build_cohort_lead_summary(leads_df, source_df, episodes_df, analysis_end=None, months=6):
    analysis_end = pd.Timestamp(analysis_end or datetime.now()).normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    window_start = analysis_end - pd.DateOffset(months=months)
    cols = resolve_lead_flag_columns(leads_df)
    wa_source_ids = build_whatsapp_source_ids(source_df)
    source_name_map = {}
    name_col = _pick_col(source_df, "name")
    id_col = _pick_col(source_df, "_id")
    if name_col and id_col:
        source_name_map = dict(zip(source_df[id_col].astype(str).str.strip(), source_df[name_col].astype(str)))

    lead_col = cols["lead_id"] or "lead"
    leads = leads_df.copy()
    leads["lead_clean"] = leads[lead_col].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    ep = episodes_df.copy()
    ep["lead_clean"] = ep["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    episode_leads = set(ep["lead_clean"].unique())
    rows = []
    for lead_clean, lead_grp in leads.groupby("lead_clean"):
        if lead_clean not in episode_leads:
            continue
        lead_row = lead_grp.iloc[0]
        lead_eps = ep[ep["lead_clean"] == lead_clean].sort_values("episode_number")
        # OR logic: any qualifying Orange / Purple / WhatsApp event in window is enough
        events = extract_lead_qualifying_events(lead_row, cols, wa_source_ids, source_name_map, window_start, analysis_end, lead_eps)
        if not events:
            continue
        ev_df = pd.DataFrame(events)
        channels = sorted(ev_df["channel"].unique().tolist())
        born_channels = sorted(ev_df.loc[ev_df["event_type"] == "born", "channel"].unique().tolist())
        react_channels = sorted(ev_df.loc[ev_df["event_type"] == "reactivated", "channel"].unique().tolist())
        resub_channels = sorted(ev_df.loc[ev_df["event_type"] == "resubmitted", "channel"].unique().tolist())
        rows.append({
            "lead": lead_clean,
            "qualifying_channels": "|".join(channels),
            "qualifying_event_types": "|".join(sorted(ev_df["event_type"].unique().tolist())),
            "original_source": born_channels[0] if born_channels else channels[0],
            "qualifying_source": "|".join(channels),
            "reactivation_source": "|".join(react_channels) if react_channels else "",
            "resubmission_source": "|".join(resub_channels) if resub_channels else "",
            "is_orange_lead": int("Orange" in channels),
            "is_purple_lead": int("Purple" in channels),
            "is_whatsapp_lead": int("WhatsApp" in channels),
            "qualifying_event_count": len(events),
            "first_qualifying_event_time": ev_df["event_time"].min(),
            "last_qualifying_event_time": ev_df["event_time"].max(),
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=[
        "lead", "qualifying_channels", "qualifying_event_types", "original_source",
        "qualifying_source", "reactivation_source", "resubmission_source",
        "is_orange_lead", "is_purple_lead", "is_whatsapp_lead",
        "qualifying_event_count", "first_qualifying_event_time", "last_qualifying_event_time",
    ])

def enrich_episode_dataset(df_master, cohort_summary):
    df = df_master.copy()
    df["lead_clean"] = df["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    df = df.merge(cohort_summary.rename(columns={"lead": "lead_clean"}), on="lead_clean", how="inner")
    df["episode_group"] = np.where(df["episode_number"] >= 5, "Episode 5+", "Episode " + df["episode_number"].astype(int).astype(str))
    if "total_attempted_calls" in df.columns and "total_answered_calls" in df.columns and "pickup_ratio" not in df.columns:
        df["pickup_ratio"] = np.where(df["total_attempted_calls"] > 0, df["total_answered_calls"] / df["total_attempted_calls"], 0.0)
    if {"total_web_timespent", "active_session_count"}.issubset(df.columns) and "avg_web_activity" not in df.columns:
        df["avg_web_activity"] = np.where(df["active_session_count"] > 0, df["total_web_timespent"] / df["active_session_count"], 0.0)
    df["stage_transition_path"] = (
        df["start_stage"].astype(str).str.strip() if "start_stage" in df.columns else ""
    ) + " -> " + (
        df["stage_before_ni"].astype(str).str.strip() if "stage_before_ni" in df.columns else ""
    )
    return df.drop(columns=["lead_clean"], errors="ignore")

print("OPW cohort helper functions loaded.")





OPW cohort helper functions loaded.


In [20]:
# Step 1: candidate leads from Athena (6 month event window, OR logic at row level)
wa_ids = set(
    _leadSourceDF.loc[_leadSourceDF["name"].astype(str).str.contains("whatsapp", case=False, na=False), "_id"]
    .astype(str)
    .str.strip()
)
wa_sql = ", ".join(f"'{x}'" for x in wa_ids) if wa_ids else "''"

# Athena leads dump uses lowercase names. There is no isorange column.
# Orange/Purple come from timestamps (+ ispurplelead when present).
leads_desc = athena_read(
    f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = '{ATHENA_DB}'
      AND table_name = 'leads'
    """
)
lead_cols = {str(c).lower(): str(c) for c in leads_desc.iloc[:, 0]}


def _sql_col(*names):
    for n in names:
        if n.lower() in lead_cols:
            return lead_cols[n.lower()]
    return None


orange_time_col = _sql_col("orangeleadtime", "orange_lead_time", "orangeLeadTime")
purple_time_col = _sql_col("purpleleadtime", "purple_lead_time", "purpleLeadTime")
orange_flag_col = _sql_col("isorange", "is_orange", "isOrange")
purple_flag_col = _sql_col("ispurplelead", "is_purple", "ispurple", "isPurple")
born_col = _sql_col("bornsource", "bornSource")
sec_col = _sql_col("secondarysource", "secondarySource")
created_col = _sql_col("createdat", "createdAt")
updated_col = _sql_col("updatedat", "updatedAt")
gid_col = _sql_col("globalid", "globalId")
project_col = _sql_col("project")
status_col = _sql_col("status")
source_col = _sql_col("source")
leadid_col = _sql_col("leadid", "leadId")
masterleadid_col = _sql_col("masterleadid", "masterLeadId")

selects = ["_id"]
if born_col:
    selects.append(f"{born_col} AS bornSource")
if source_col:
    selects.append(f"{source_col} AS source")
if sec_col:
    selects.append(f"{sec_col} AS secondarySource")
if created_col:
    selects.append(f"{created_col} AS lead_createdAt")
if orange_flag_col:
    selects.append(f"{orange_flag_col} AS is_orange")
if purple_flag_col:
    selects.append(f"{purple_flag_col} AS is_purple")
if orange_time_col:
    selects.append(f"{orange_time_col} AS orange_lead_time")
if purple_time_col:
    selects.append(f"{purple_time_col} AS purple_lead_time")
if gid_col:
    selects.append(f"{gid_col} AS globalId")
if project_col:
    selects.append(f"{project_col} AS project")
if status_col:
    selects.append(f"{status_col} AS status")
if leadid_col:
    selects.append(f"{leadid_col} AS leadId")
if masterleadid_col:
    selects.append(f"{masterleadid_col} AS masterLeadId")

where_parts = []
if orange_time_col:
    where_parts.append(f"{orange_time_col} >= TIMESTAMP '{WINDOW_START_STR}'")
if purple_time_col:
    where_parts.append(f"{purple_time_col} >= TIMESTAMP '{WINDOW_START_STR}'")
if born_col and created_col:
    where_parts.append(f"({born_col} IN ({wa_sql}) AND {created_col} >= TIMESTAMP '{WINDOW_START_STR}')")
if updated_col:
    where_parts.append(f"{updated_col} >= TIMESTAMP '{WINDOW_START_STR}'")
if not where_parts:
    raise RuntimeError("Could not find orange/purple/whatsapp columns on leads")

candidate_sql = f"""
SELECT
    {", ".join(selects)}
FROM "{ATHENA_DB}"."leads"
WHERE {" OR ".join(where_parts)}
"""
leads_candidates = athena_read(candidate_sql)
print(f"Candidate leads from Athena: {len(leads_candidates):,}")
print("Lead columns used:", ", ".join(selects))

# Step 2: refine with Python OR rules (Orange / Purple / WhatsApp)
source_name_map = dict(zip(_leadSourceDF["_id"].astype(str).str.strip(), _leadSourceDF["name"].astype(str)))
wa_source_ids = wa_ids
cols = resolve_lead_flag_columns(leads_candidates)

qualifying_leads = []
for lead_clean, grp in leads_candidates.assign(
    lead_clean=leads_candidates["_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
).groupby("lead_clean"):
    events = extract_lead_qualifying_events(
        grp.iloc[0], cols, wa_source_ids, source_name_map, WINDOW_START, ANALYSIS_END, episodes=None
    )
    if events:
        qualifying_leads.append(lead_clean)

qualifying_leads = sorted(set(qualifying_leads))
print(f"Qualifying OPW leads (OR filter): {len(qualifying_leads):,}")


Candidate leads from Athena: 1,249,169
Lead columns used: _id, bornsource AS bornSource, source AS source, secondarysource AS secondarySource, createdat AS lead_createdAt, ispurplelead AS is_purple, orangeleadtime AS orange_lead_time, purpleleadtime AS purple_lead_time, globalid AS globalId, project AS project, status AS status, leadid AS leadId, masterleadid AS masterLeadId
Qualifying OPW leads (OR filter): 34,736


In [21]:
# Step 3: full stage history and episodes for qualifying leads only
_leadsStageDF = athena_batches(
    """
    SELECT _id, lead, stage, iscurrent AS isCurrent, createdat AS createdAt,
           reason, isstagemovement AS isStageMovement, cancelledreason AS cancellation_reason, comments
    FROM "{db}"."leadstagehistories"
    WHERE lead IN ({{ids}})
    """.format(db=ATHENA_DB),
    qualifying_leads,
)

df_hist = _leadsStageDF.copy()
df_hist["createdAt_dt"] = pd.to_datetime(df_hist["createdAt"], errors="coerce")
df_hist["lead_clean"] = df_hist["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
df_hist = df_hist.dropna(subset=["lead_clean", "createdAt_dt"]).sort_values(["lead_clean", "createdAt_dt"]).reset_index(drop=True)
df_hist["prev_stage"] = df_hist.groupby("lead_clean")["stage"].shift(1)

is_duplicate_ni = (df_hist["stage"] == "Not Interested") & (df_hist["prev_stage"] == "Not Interested")
df_valid = df_hist[~is_duplicate_ni].copy().reset_index(drop=True)
df_valid["prev_stage"] = df_valid.groupby("lead_clean")["stage"].shift(1)
df_valid["is_first_ni"] = (df_valid["stage"] == "Not Interested") & (df_valid["prev_stage"] != "Not Interested")
df_valid["is_last_rec"] = df_valid["lead_clean"] != df_valid["lead_clean"].shift(-1)
df_valid["is_ep_end"] = df_valid["is_first_ni"] | df_valid["is_last_rec"]
df_valid["ep_end_shift"] = df_valid.groupby("lead_clean")["is_ep_end"].shift(1).fillna(False).astype(int)
df_valid["episode_number"] = df_valid.groupby("lead_clean")["ep_end_shift"].cumsum() + 1

df_episodes = df_valid.groupby(["lead_clean", "episode_number"]).agg(
    episode_start_time=("createdAt_dt", "min"),
    episode_end_time=("createdAt_dt", "max"),
    start_stage=("stage", "first"),
    end_stage=("stage", "last"),
    total_stage_movements=("createdAt_dt", "count"),
).reset_index().rename(columns={"lead_clean": "lead"})
df_episodes["episode_id"] = df_episodes["lead"] + "_" + df_episodes["episode_number"].astype(str)
max_ep = df_episodes.groupby("lead")["episode_number"].transform("max")
df_episodes["is_reactivated_episode"] = (df_episodes["episode_number"] < max_ep).astype(int)

leads_for_cohort = leads_candidates[
    leads_candidates["_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip().isin(qualifying_leads)
].copy()
df_episodes["lead_clean"] = df_episodes["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
leads_for_cohort["lead_clean"] = leads_for_cohort["_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
df5 = df_episodes.merge(leads_for_cohort, on="lead_clean", how="left", suffixes=("", "_lead"))
df5.drop(columns=["lead_clean"], errors="ignore", inplace=True)
print(f"Episode rows (all episodes for qualifying leads): {len(df5):,}")


Episode rows (all episodes for qualifying leads): 45,115


In [22]:
# Step 4: web, stage, whatsapp, call features (no 3 month lead-created filter)
# globalId -> visitorId
if "globalId" in df5.columns:
    gids = df5["globalId"].dropna().astype(str).str.strip().unique().tolist()
    df3 = athena_batches(
        f'SELECT globalid AS globalId, visitorid AS visitorId, project FROM "{ATHENA_DB}"."globalvisitorlogs" WHERE globalid IN ({{ids}})',
        gids,
    )
    if not df3.empty and {"project", "globalId", "visitorId"}.issubset(df3.columns):
        df5 = df5.merge(df3, on=["project", "globalId"], how="left")
    elif "visitorId" not in df5.columns:
        df5["visitorId"] = np.nan
else:
    df5["visitorId"] = np.nan

# visitor logs for all qualifying visitor IDs (episode windows applied later)
visitor_ids = df5["visitorId"].dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.strip().unique().tolist()
df6_parts = []
for i in range(0, len(visitor_ids), 500):
    batch = visitor_ids[i : i + 500]
    ids = ", ".join(f"'{v}'" for v in batch)
    q = f"""
        SELECT _id, visitorid AS visitorId, globalid AS globalId,
               cummulativetimespent AS cummulativeTimeSpent, createdat AS createdAt,
               instancenumber AS instanceNumber, pages, pricesheetsopened AS priceSheetsOpened,
               whatsappclicked AS whatsappClicked
        FROM "{ATHENA_DB}"."visitorlogs"
        WHERE visitorid IN ({ids})
    """
    df6_parts.append(athena_read(q))
df6 = pd.concat(df6_parts, ignore_index=True) if df6_parts else pd.DataFrame()

# track events
if not df6.empty:
    botlogs = df6["_id"].dropna().astype(str).str.replace(r"\.0$", "", regex=True).str.strip().unique().tolist()
    df7 = athena_batches(f'SELECT botlog AS botLog, trackevents AS trackEvents FROM "{ATHENA_DB}"."userredirecttracks" WHERE botlog IN ({{ids}})', botlogs)
    df6 = df6.merge(df7, left_on="_id", right_on="botLog", how="left")
else:
    df7 = pd.DataFrame()

# parse pages + click flags (same rules as leadstagehistories)
def parse_pages_json(val):
    d = {f"{p}_timespent": 0.0 for p in ["home","price","plan","location","amenities","media","specifications"]}
    if isinstance(val, str) and val.strip():
        try:
            p = json.loads(val)
            if isinstance(p, dict):
                for k, v in p.items():
                    key = f"{k.lower().strip()}_timespent"
                    if key in d:
                        d[key] = float(v) if pd.notnull(v) else 0.0
        except Exception:
            pass
    return d

def parse_track_events_json(val):
    flags = {k: 0 for k in ["is_site_visit_book_clicked","is_whatsapp_clicked","is_brochure_downloaded","is_price_sheet_viewed","is_cost_sheet_clicked"]}
    if isinstance(val, str) and val.strip():
        try:
            val = json.loads(val)
        except Exception:
            val = []
    if not isinstance(val, list):
        return flags
    for item in val:
        if not isinstance(item, dict):
            continue
        evs = item.get("eventName")
        if isinstance(evs, str):
            evs = [evs]
        for ev in evs or []:
            u = str(ev).upper()
            if any(k in u for k in ["BOOK_A_SITE_VISIT","SITE_VISIT_BTN","START_SITE_VISIT"]):
                flags["is_site_visit_book_clicked"] = 1
            if any(k in u for k in ["WHATSAPP","WA_"]):
                flags["is_whatsapp_clicked"] = 1
            if any(k in u for k in ["BROCHURE","DOWNLOAD","PDF"]):
                flags["is_brochure_downloaded"] = 1
            if any(k in u for k in ["PRICE_SHEET","PRICE_SECTION","PRICESHEET","PRICE_PARTICULARS","UNLOCK_PRICE"]):
                flags["is_price_sheet_viewed"] = 1
            if any(k in u for k in ["COST_SHEET","SHOW_COST_SHEET","VIEW_COST_SHEET"]):
                flags["is_cost_sheet_clicked"] = 1
    return flags

if not df6.empty:
    pages_df = pd.DataFrame(df6["pages"].apply(parse_pages_json).tolist())
    df6_populated = pd.concat([df6.reset_index(drop=True), pages_df.reset_index(drop=True)], axis=1)
    ev_df = pd.DataFrame(df6["trackEvents"].apply(parse_track_events_json).tolist()) if "trackEvents" in df6.columns else pd.DataFrame()
    if not ev_df.empty:
        df6_populated = pd.concat([df6_populated, ev_df], axis=1)
    for c in [x for x in df6_populated.columns if x.startswith("is_")]:
        df6_populated[c] = df6_populated.get(c, 0).fillna(0).astype(int)
else:
    df6_populated = pd.DataFrame()

# episode start adjustment for reactivated episodes
df5["episode_start_time"] = pd.to_datetime(df5["episode_start_time"], errors="coerce")
df5["episode_end_time"] = pd.to_datetime(df5["episode_end_time"], errors="coerce")
df5 = df5.sort_values(["lead", "episode_number"]).reset_index(drop=True)
df5["prev_ep_end_time"] = df5.groupby("lead")["episode_end_time"].shift(1)
micro_offset = pd.Timedelta(milliseconds=1)
df5["episode_start_time"] = np.where(
    df5["episode_number"] > 1,
    df5["prev_ep_end_time"] + micro_offset,
    df5["episode_start_time"],
)
df5.drop(columns=["prev_ep_end_time"], errors="ignore", inplace=True)

# web aggregation per episode window
if not df6_populated.empty:
    base = df5.copy()
    base["visitorId_clean"] = base["visitorId"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    pop = df6_populated.copy()
    pop["createdAt_dt"] = pd.to_datetime(pop["createdAt"], errors="coerce")
    pop["visitorId_clean"] = pop["visitorId"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    m = base[["episode_id","visitorId_clean","episode_start_time","episode_end_time"]].merge(pop, on="visitorId_clean", how="inner")
    v = m[(m["createdAt_dt"] >= m["episode_start_time"]) & (m["createdAt_dt"] <= m["episode_end_time"])].copy()
    if not v.empty:
        v["cummulativeTimeSpent"] = pd.to_numeric(v.get("cummulativeTimeSpent", 0), errors="coerce").fillna(0)
        v["instanceNumber"] = pd.to_numeric(v.get("instanceNumber", 0), errors="coerce").fillna(0).astype(int)
        page_cols = [c for c in v.columns if c.endswith("_timespent")]
        flag_cols = [c for c in ["is_site_visit_book_clicked","is_whatsapp_clicked","is_brochure_downloaded","is_price_sheet_viewed","is_cost_sheet_clicked"] if c in v.columns]
        agg = {"total_web_timespent": ("cummulativeTimeSpent","sum"), "instance_min": ("instanceNumber","min"), "instance_max": ("instanceNumber","max"), "active_session_count": ("cummulativeTimeSpent", lambda s: int((s>0).sum()))}
        for c in page_cols:
            agg[c] = (c, "sum")
        for c in flag_cols:
            agg[c] = (c, lambda s: int((s>0).any()))
        web_ep = v.groupby("episode_id").agg(**agg).reset_index()
        web_ep["total_session_count"] = web_ep["instance_max"] - web_ep["instance_min"] + 1
        web_ep.drop(columns=["instance_min","instance_max"], inplace=True)
        df5 = base.merge(web_ep, on="episode_id", how="left")
    else:
        df5 = base
else:
    pass

for c in ["total_web_timespent","total_session_count","active_session_count"] + [x for x in df5.columns if x.endswith("_timespent")]:
    if c in df5.columns:
        df5[c] = df5[c].fillna(0)

# resubmissions
def calculate_episode_resubmissions(row):
    cnt = 0
    val = row.get("secondarySource")
    st = pd.to_datetime(row.get("episode_start_time"), errors="coerce")
    en = pd.to_datetime(row.get("episode_end_time"), errors="coerce")
    if pd.isna(st) or pd.isna(en) or not isinstance(val, str) or val.strip() in ("", "[]"):
        return 0
    try:
        parsed = json.loads(val)
    except Exception:
        return 0
    if not isinstance(parsed, list):
        return 0
    for item in parsed:
        if not isinstance(item, dict):
            continue
        is_react = item.get("isReactivation")
        if is_react is True or str(is_react).lower() == "true":
            continue
        dt = pd.to_datetime(item.get("date"), errors="coerce")
        if pd.notna(dt) and st <= dt <= en:
            cnt += 1
    return cnt

df5["resubmission_count_in_episode"] = df5.apply(calculate_episode_resubmissions, axis=1)
df5["is_resubmitted_in_episode"] = (df5["resubmission_count_in_episode"] > 0).astype(int)

# stage metrics from full stage history
base = df5.copy()
base["lead_clean"] = base["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
stg = _leadsStageDF.copy()
stg["createdAt_dt"] = pd.to_datetime(stg["createdAt"], errors="coerce")
stg["lead_clean"] = stg["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
m = base[["episode_id","lead_clean","episode_start_time","episode_end_time"]].merge(stg, on="lead_clean", how="inner")
v = m[(m["createdAt_dt"] >= m["episode_start_time"]) & (m["createdAt_dt"] <= m["episode_end_time"])].copy()
if not v.empty:
    v["is_site_visit_reason"] = v["reason"].astype(str).str.strip().str.lower().isin(["visited site","virtual site visit"])
    sv = v.groupby("episode_id")["is_site_visit_reason"].any().astype(int).reset_index(name="did_reach_site_visit_in_episode")
    v["is_mvt"] = v["isStageMovement"].astype(str).str.strip().str.lower().isin(["true","1"])
    mv = v.groupby("episode_id")["is_mvt"].sum().astype(int).reset_index(name="episode_stage_movement_count")
    sn = v[v["stage"].astype(str).str.strip() != "Not Interested"].groupby("episode_id")["stage"].last().reset_index(name="stage_before_ni")
    base = base.merge(sv, on="episode_id", how="left").merge(mv, on="episode_id", how="left").merge(sn, on="episode_id", how="left")
base["episode_duration_days"] = (base["episode_end_time"] - base["episode_start_time"]).dt.total_seconds() / 86400.0
if "did_reach_site_visit_in_episode" not in base.columns:
    base["did_reach_site_visit_in_episode"] = 0
if "episode_stage_movement_count" not in base.columns:
    base["episode_stage_movement_count"] = 0
if "stage_before_ni" not in base.columns:
    base["stage_before_ni"] = base["start_stage"] if "start_stage" in base.columns else ""
base["did_reach_site_visit_in_episode"] = pd.to_numeric(base["did_reach_site_visit_in_episode"], errors="coerce").fillna(0).astype(int)
base["episode_stage_movement_count"] = pd.to_numeric(base["episode_stage_movement_count"], errors="coerce").fillna(0).astype(int)

def resolve_episode_source_id(row):
    ep_num = row.get("episode_number")
    born = str(row.get("bornSource", "")).strip()
    start_t = pd.to_datetime(row.get("episode_start_time"), errors="coerce")
    val = row.get("secondarySource")
    if ep_num == 1 or pd.isna(start_t):
        return born or "Unknown"
    if isinstance(val, str) and val.strip() and val != "[]":
        try:
            parsed = json.loads(val)
            if isinstance(parsed, list):
                for item in reversed(parsed):
                    if not isinstance(item, dict):
                        continue
                    is_react = item.get("isReactivation")
                    if is_react is True or str(is_react).lower() == "true":
                        rec_date = pd.to_datetime(item.get("date"), errors="coerce")
                        if pd.notna(rec_date) and abs((rec_date - start_t).total_seconds()) <= 86400:
                            return str(item.get("source", "")).strip()
        except Exception:
            pass
    return born

base["episode_source"] = base.apply(resolve_episode_source_id, axis=1)

# whatsapp in episode
wa_parts = athena_batches(
    f'SELECT lead AS lead_oid, time FROM "{ATHENA_DB}"."whatsappchatmessages" WHERE (isuser = \'true\' OR isuser = \'True\' OR isuser = \'1\') AND lead IN ({{ids}})',
    qualifying_leads,
)
if not wa_parts.empty:
    wa_parts["time_dt"] = pd.to_datetime(wa_parts["time"], errors="coerce")
    wa_parts["lead_clean"] = wa_parts["lead_oid"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    bw = base.copy()
    bw["lead_clean"] = bw["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    m = bw[["episode_id","lead_clean","episode_start_time","episode_end_time"]].merge(wa_parts, on="lead_clean", how="inner")
    v = m[(m["time_dt"] >= m["episode_start_time"]) & (m["time_dt"] <= m["episode_end_time"])]
    if not v.empty:
        wa_ep = v.groupby("episode_id").agg(
            episode_whatsapp_inbound_count=("time_dt","count"),
            first_whatsapp_time=("time_dt","min"),
            last_whatsapp_time=("time_dt","max"),
        ).reset_index()
        base = base.merge(wa_ep, on="episode_id", how="left")
if "episode_whatsapp_inbound_count" not in base.columns:
    base["episode_whatsapp_inbound_count"] = 0
base["episode_whatsapp_inbound_count"] = pd.to_numeric(base["episode_whatsapp_inbound_count"], errors="coerce").fillna(0).astype(int)
base["has_whatsapp_inbound_in_episode"] = (base["episode_whatsapp_inbound_count"] > 0).astype(int)
if "first_whatsapp_time" in base.columns:
    base["days_to_first_whatsapp_inbound"] = ((base["first_whatsapp_time"] - base["episode_start_time"]).dt.total_seconds() / 86400.0).fillna(-1.0)
    base["days_to_last_whatsapp_inbound"] = ((base["last_whatsapp_time"] - base["episode_start_time"]).dt.total_seconds() / 86400.0).fillna(-1.0)
    base.drop(columns=["first_whatsapp_time","last_whatsapp_time"], errors="ignore", inplace=True)
else:
    base["days_to_first_whatsapp_inbound"] = -1.0
    base["days_to_last_whatsapp_inbound"] = -1.0

# calls: full history for qualifying leads (not 3 month creation filter)
_callsDF = athena_batches(
    f'SELECT lead, createdat AS createdAt, direction, callstatus AS status, callduration AS duration FROM "{ATHENA_DB}"."leadcallrecords" WHERE lead IN ({{ids}})',
    qualifying_leads,
)
call_cols = [
    "lead", "episode_number", "total_answered_calls", "total_inbound_calls",
    "total_outbound_calls", "total_agent_missed_calls", "total_customer_missed_calls",
    "total_attempted_calls", "total_call_duration",
]
if _callsDF.empty:
    call_stats = pd.DataFrame(columns=call_cols)
else:
    _calls = _callsDF.copy()
    _calls["lead"] = _calls["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    _calls["call_time"] = pd.to_datetime(_calls["createdAt"], errors="coerce")
    _calls["call_status"] = _calls["status"].astype(str).str.strip().str.lower()
    _calls["call_direction"] = _calls["direction"].astype(str).str.strip().str.lower()
    _calls["call_duration"] = pd.to_numeric(_calls["duration"], errors="coerce").fillna(0)
    _eps = base.copy()
    _eps["lead"] = _eps["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    merged = _calls.merge(_eps[["lead","episode_number","episode_start_time","episode_end_time"]], on="lead", how="inner")
    merged = merged[(merged["call_time"] >= merged["episode_start_time"]) & (merged["call_time"] <= merged["episode_end_time"])].copy()
    if merged.empty:
        call_stats = pd.DataFrame(columns=call_cols)
    else:
        merged["is_answered"] = merged["call_status"].eq("answered")
        merged["is_inbound"] = merged["call_direction"].eq("inbound")
        merged["is_outbound"] = merged["call_direction"].eq("outbound")
        merged["is_agent_missed"] = merged["call_status"].eq("agent-missed")
        merged["is_customer_missed"] = merged["call_status"].eq("customer-missed")
        merged["is_attempted"] = merged["call_status"].isin(["answered","customer-missed"])
        merged["answered_duration"] = np.where(merged["is_answered"], merged["call_duration"], 0.0)
        call_stats = merged.groupby(["lead","episode_number"], as_index=False).agg(
            total_answered_calls=("is_answered","sum"),
            total_inbound_calls=("is_inbound","sum"),
            total_outbound_calls=("is_outbound","sum"),
            total_agent_missed_calls=("is_agent_missed","sum"),
            total_customer_missed_calls=("is_customer_missed","sum"),
            total_attempted_calls=("is_attempted","sum"),
            total_call_duration=("answered_duration","sum"),
        )
df_master_episode_dataset = base.merge(call_stats, on=["lead","episode_number"], how="left")
for c in ["total_answered_calls","total_inbound_calls","total_outbound_calls","total_agent_missed_calls","total_customer_missed_calls","total_attempted_calls"]:
    if c not in df_master_episode_dataset.columns:
        df_master_episode_dataset[c] = 0
    df_master_episode_dataset[c] = df_master_episode_dataset[c].fillna(0).astype(int)
if "total_call_duration" not in df_master_episode_dataset.columns:
    df_master_episode_dataset["total_call_duration"] = 0.0
df_master_episode_dataset["total_call_duration"] = df_master_episode_dataset["total_call_duration"].fillna(0.0)
df_master_episode_dataset = replace_ids_with_names(df_master_episode_dataset)
print(f"Master episode dataset: {len(df_master_episode_dataset):,} rows, {df_master_episode_dataset['lead'].nunique():,} leads")
for col in ["project", "bornSource", "source", "episode_source"]:
    if col in df_master_episode_dataset.columns:
        print(f"  {col}: {df_master_episode_dataset[col].nunique(dropna=True):,} unique  sample: {df_master_episode_dataset[col].dropna().astype(str).unique()[:6].tolist()}")


Master episode dataset: 45,115 rows, 34,735 leads
  project: 8 unique  sample: ['SPECTRA', 'SPRINGS', 'SPIRE', 'LOFT', 'LANDMARK', 'BROADWAY']
  bornSource: 32 unique  sample: ['Channel Partner', 'Web Lead', 'Phone', 'Call', 'magic-bricks', 'Social Media']
  source: 33 unique  sample: ['ASBL DB', 'Web Lead', 'Phone', 'Call', 'magic-bricks', 'Social Media']
  episode_source: 33 unique  sample: ['Channel Partner', 'Web Lead', '99-acres', 'Phone', 'Call', 'magic-bricks']


In [23]:
def feature_columns(df):
    cols = []
    for c in df.columns:
        if c in META_EXCLUDE or c.endswith(("_curr", "_prev")):
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def pct_change_label(prev, curr):
    if pd.isna(prev) or pd.isna(curr):
        return "Insufficient data"
    if prev == 0 and curr > 0:
        return "New Activity"
    if prev == 0 and curr == 0:
        return "No Activity"
    return f"{((curr - prev) / prev) * 100:.2f}%"

def classify_trend(values):
    clean = [v for v in values if pd.notna(v)]
    if len(clean) < 2:
        return "Insufficient data"
    deltas = [b - a for a, b in zip(clean, clean[1:])]
    if all(d > 0 for d in deltas):
        return "Increasing"
    if all(d < 0 for d in deltas):
        return "Decreasing"
    if all(abs(d) <= max(1e-9, 0.01 * abs(a)) for a, d in zip(clean, deltas)):
        return "Stable"
    return "Mixed"

# ponytail: drop <=0 then Tukey 1.5 IQR; 0/1 flags stay as rates. Upgrade to MAD if a feature stays heavy-tailed
def active_values(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return s[s > 0]

def is_binary_feature(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return False
    return set(s.unique()).issubset({0, 1, 0.0, 1.0})

def without_outliers(s):
    s = active_values(s)
    if len(s) < 4:
        return s
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return s
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s >= lo) & (s <= hi)]

def clean_mean(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.nan
    if is_binary_feature(s):
        return float(s.mean())
    s = without_outliers(s)
    return float(s.mean()) if len(s) else np.nan

def episode_average_table(df, features=None):
    features = list(features or feature_columns(df))
    out = pd.DataFrame({"Feature": features})
    for ep in EPISODE_ORDER:
        out[ep] = [clean_mean(df.loc[df["episode_group"] == ep, f]) if f in df.columns else np.nan for f in features]
    return out

def episode_change_table(episode_avgs):
    rows = []
    transitions = list(zip(EPISODE_ORDER, EPISODE_ORDER[1:]))
    for _, row in episode_avgs.iterrows():
        rec = {"Feature": row["Feature"], "Overall Trend": classify_trend([row[e] for e in EPISODE_ORDER])}
        for ep in EPISODE_ORDER:
            rec[f"{ep} Avg"] = row[ep]
        for prev_ep, curr_ep in transitions:
            prev, curr = row[prev_ep], row[curr_ep]
            rec[f"Δ {curr_ep.split()[-1]}→{prev_ep.split()[-1]}"] = curr - prev if pd.notna(prev) and pd.notna(curr) else np.nan
            rec[f"% {curr_ep.split()[-1]}→{prev_ep.split()[-1]}"] = pct_change_label(prev, curr)
        rows.append(rec)
    return pd.DataFrame(rows)

def episode_sample_size_table(df):
    return pd.DataFrame([
        {"Episode": ep, "Unique Leads": df.loc[df["episode_group"] == ep, "lead"].nunique(), "Total Episodes": len(df[df["episode_group"] == ep])}
        for ep in EPISODE_ORDER
    ])

def source_comparison_table(episode_avgs_by_source):
    if not episode_avgs_by_source:
        return pd.DataFrame()
    base = None
    for source, tbl in episode_avgs_by_source.items():
        if tbl is None or tbl.empty:
            continue
        part = tbl.set_index("Feature").rename(columns={ep: f"{source} {ep}" for ep in EPISODE_ORDER})
        base = part if base is None else base.join(part, how="outer")
    return base.reset_index() if base is not None else pd.DataFrame()

def source_overall_average_table(df, features):
    rows = []
    for f in features:
        row = {"Feature": f, "Aggregation Level": "mean of values > 0 after dropping IQR outliers"}
        for source, flag in [("Orange", "is_orange_lead"), ("Purple", "is_purple_lead"), ("WhatsApp", "is_whatsapp_lead")]:
            sub = df[df[flag] == 1] if flag in df.columns else df.iloc[0:0]
            row[f"{source} Overall Avg"] = clean_mean(sub[f]) if f in sub.columns and len(sub) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def participation_table(df, features):
    rows = []
    for f in features:
        if f not in df.columns:
            continue
        non, react = df[df["is_reactivated_episode"] == 0], df[df["is_reactivated_episode"] == 1]
        p_non = (non[f] > 0).mean() * 100 if len(non) else np.nan
        p_rec = (react[f] > 0).mean() * 100 if len(react) else np.nan
        rows.append({"Feature": f, "Non-Reactivated % Active": p_non, "Reactivated % Active": p_rec, "Difference (pp)": p_rec - p_non})
    return pd.DataFrame(rows)

def conditional_intensity_table(df, features):
    rows = []
    for f in features:
        if f not in df.columns:
            continue
        for label, mask in [("Non-Reactivated", df["is_reactivated_episode"] == 0), ("Reactivated", df["is_reactivated_episode"] == 1)]:
            s = without_outliers(df.loc[mask, f])
            rows.append({
                "Feature": f, "Cohort": label, "Active Episodes": len(s),
                "Mean": s.mean() if len(s) else np.nan, "Median": s.median() if len(s) else np.nan,
                "P75": s.quantile(0.75) if len(s) else np.nan, "P90": s.quantile(0.90) if len(s) else np.nan,
                "P95": s.quantile(0.95) if len(s) else np.nan,
            })
    return pd.DataFrame(rows)

def reactivation_comparison_table(df, features):
    rows = []
    non, react = df[df["is_reactivated_episode"] == 0], df[df["is_reactivated_episode"] == 1]
    for f in features:
        if f not in df.columns:
            continue
        m_non, m_rec = clean_mean(non[f]), clean_mean(react[f])
        rows.append({"Feature": f, "Non-Reactivated Avg": m_non, "Reactivated Avg": m_rec, "Absolute Change": m_rec - m_non, "% Change": pct_change_label(m_non, m_rec)})
    return pd.DataFrame(rows)

def build_within_lead_pairs(df):
    ep = df.sort_values(["lead", "episode_number"]).copy()
    ep["lead_clean"] = ep["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    curr = ep[ep["episode_number"] >= 2].copy()
    curr["prev_episode_number"] = curr["episode_number"] - 1
    prev = ep.copy()
    return curr.merge(prev, left_on=["lead_clean", "prev_episode_number"], right_on=["lead_clean", "episode_number"], suffixes=("_curr", "_prev"))

def within_lead_pair_table(df_pairs, features):
    rows = []
    pair_labels = {2: "2 -> 1", 3: "3 -> 2", 4: "4 -> 3"}
    label_pair = lambda n: pair_labels.get(int(n), "5+ -> 4+")
    for f in features:
        prev_col, curr_col = f"{f}_prev", f"{f}_curr"
        if prev_col not in df_pairs.columns or curr_col not in df_pairs.columns:
            continue
        for pair_name, sub in df_pairs.groupby(df_pairs["episode_number_curr"].map(label_pair)):
            prev = pd.to_numeric(sub[prev_col], errors="coerce")
            curr = pd.to_numeric(sub[curr_col], errors="coerce")
            if len(prev) == 0:
                continue
            delta = curr.fillna(0.0) - prev.fillna(0.0)
            p_val = 1.0
            if len(prev.dropna()) > 1 and not (delta == 0).all():
                try:
                    p_val = stats.ttest_rel(curr.fillna(0.0), prev.fillna(0.0)).pvalue
                except Exception:
                    p_val = 1.0
            prev_avg, curr_avg = clean_mean(prev), clean_mean(curr)
            rows.append({
                "pair": pair_name, "feature": f, "active_pair_count": int(len(prev)),
                "previous_average": prev_avg, "current_average": curr_avg,
                "mean_shift": (curr_avg - prev_avg) if pd.notna(prev_avg) and pd.notna(curr_avg) else np.nan,
                "% change": pct_change_label(prev_avg, curr_avg),
                "% increased": (delta > 0).mean() * 100, "% unchanged": (delta == 0).mean() * 100,
                "% decreased": (delta < 0).mean() * 100, "test_used": "paired t-test",
                "p_value": p_val, "significance": "p<0.05" if p_val < 0.05 else "NS",
            })
    return pd.DataFrame(rows)

def stage_transition_table(df):
    if "stage_transition_path" not in df.columns:
        return pd.DataFrame()
    return pd.crosstab(df["stage_transition_path"], df["episode_group"], normalize="index").mul(100).reset_index()

def site_visit_analysis_table(df):
    cols = [c for c in ["did_reach_site_visit_in_episode", "is_site_visit_book_clicked"] if c in df.columns]
    if not cols:
        return pd.DataFrame()
    rows = []
    for ep in EPISODE_ORDER:
        sub = df[df["episode_group"] == ep]
        row = {"Episode": ep, "Episodes": len(sub)}
        for c in cols:
            row[f"{c} avg"] = clean_mean(sub[c])
            row[f"{c} % active"] = (sub[c] > 0).mean() * 100
        rows.append(row)
    return pd.DataFrame(rows)

def executive_summary(df, episode_change):
    highlights = []
    for feature in ["total_web_timespent", "total_call_duration", "episode_whatsapp_inbound_count", "resubmission_count_in_episode", "episode_stage_movement_count", "episode_duration_days"]:
        part = episode_change[episode_change["Feature"] == feature]
        if part.empty:
            continue
        highlights.append({
            "Feature": feature,
            "Overall Trend": part.iloc[0]["Overall Trend"],
            "Episode 1 Avg": part.iloc[0].get("Episode 1 Avg"),
            "Episode 5+ Avg": part.iloc[0].get("Episode 5+ Avg"),
            "Interpretation": "Episode average among values > 0 after dropping IQR outliers.",
        })
    summary = pd.DataFrame({
        "metric": ["analysis_end_date", "unique_leads", "total_episodes", "orange_leads", "purple_leads", "whatsapp_leads"],
        "value": [
            pd.Timestamp.now().strftime("%Y-%m-%d"), df["lead"].nunique(), len(df),
            int((df.drop_duplicates("lead")["is_orange_lead"] == 1).sum()) if "is_orange_lead" in df.columns else np.nan,
            int((df.drop_duplicates("lead")["is_purple_lead"] == 1).sum()) if "is_purple_lead" in df.columns else np.nan,
            int((df.drop_duplicates("lead")["is_whatsapp_lead"] == 1).sum()) if "is_whatsapp_lead" in df.columns else np.nan,
        ],
    })
    return summary, pd.DataFrame(highlights)

def _format_sheet(writer, sheet_name, df):
    df.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]
    ws.freeze_panes = "A2"
    if sheet_name == "01_Full_Episode_Data":
        ws.auto_filter.ref = ws.dimensions

def display_df(df, drop_cols=None, keep_cols=None, sort_by=None, rows=None):
    if df is None or df.empty:
        return df
    out = df.copy()
    if keep_cols is not None:
        out = out[[c for c in keep_cols if c in out.columns]]
    if drop_cols is not None:
        out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors="ignore")
    if sort_by is not None:
        sort_cols = [sort_by] if isinstance(sort_by, str) else list(sort_by)
        out = out.sort_values([c for c in sort_cols if c in out.columns])
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(2)
    if rows is not None:
        out = out.head(rows)
    return out

def show_sheet(title, df, drop_cols=None, keep_cols=None, sort_by=None, rows=None):
    print("=" * 110)
    print(title)
    print("=" * 110)
    out = display_df(df, drop_cols=drop_cols, keep_cols=keep_cols, sort_by=sort_by, rows=rows)
    if out is None or out.empty:
        print("(empty)")
        return out
    with pd.option_context("display.max_rows", 200, "display.max_columns", 40, "display.width", 220):
        print(out.to_string(index=False))
    return out

print("OPW analysis table functions loaded.")




OPW analysis table functions loaded.


In [24]:
# Build cohort (OR logic), analysis tables, and Excel workbook
cohort_summary = build_cohort_lead_summary(
    leads_df=leads_for_cohort,
    source_df=_leadSourceDF,
    episodes_df=df_master_episode_dataset,
    analysis_end=ANALYSIS_END,
    months=ANALYSIS_MONTHS,
)
df_opw_cohort = enrich_episode_dataset(df_master_episode_dataset, cohort_summary)
df_opw_cohort = replace_ids_with_names(df_opw_cohort)
features = feature_columns(df_opw_cohort)

episode_avgs = episode_average_table(df_opw_cohort, features)
episode_change = episode_change_table(episode_avgs)
source_avgs = {
    source: episode_average_table(df_opw_cohort[df_opw_cohort[flag] == 1], features)
    for source, flag in [("Orange", "is_orange_lead"), ("Purple", "is_purple_lead"), ("WhatsApp", "is_whatsapp_lead")]
    if flag in df_opw_cohort.columns
}
pairs = build_within_lead_pairs(df_opw_cohort)
exec_summary, exec_highlights = executive_summary(df_opw_cohort, episode_change)

cohort_summary_df = pd.DataFrame({
    "metric": [
        "total_unique_leads", "total_episodes", "episode_1_count", "episode_2_count",
        "episode_3_count", "episode_4_count", "episode_5_plus_count",
        "orange_leads", "purple_leads", "whatsapp_leads",
    ],
    "value": [
        df_opw_cohort["lead"].nunique(), len(df_opw_cohort),
        len(df_opw_cohort[df_opw_cohort["episode_group"] == "Episode 1"]),
        len(df_opw_cohort[df_opw_cohort["episode_group"] == "Episode 2"]),
        len(df_opw_cohort[df_opw_cohort["episode_group"] == "Episode 3"]),
        len(df_opw_cohort[df_opw_cohort["episode_group"] == "Episode 4"]),
        len(df_opw_cohort[df_opw_cohort["episode_group"] == "Episode 5+"]),
        int(df_opw_cohort.drop_duplicates("lead")["is_orange_lead"].sum()) if "is_orange_lead" in df_opw_cohort.columns else 0,
        int(df_opw_cohort.drop_duplicates("lead")["is_purple_lead"].sum()) if "is_purple_lead" in df_opw_cohort.columns else 0,
        int(df_opw_cohort.drop_duplicates("lead")["is_whatsapp_lead"].sum()) if "is_whatsapp_lead" in df_opw_cohort.columns else 0,
    ],
})
episode_sample_size = episode_sample_size_table(df_opw_cohort)
source_comparison = source_comparison_table(source_avgs)
source_overall_avg = source_overall_average_table(df_opw_cohort, features)
reactivation_comparison = reactivation_comparison_table(df_opw_cohort, features)
participation = participation_table(df_opw_cohort, features)
conditional_intensity = conditional_intensity_table(df_opw_cohort, features)
within_lead_pairs = within_lead_pair_table(pairs, features)
stage_transitions = stage_transition_table(df_opw_cohort)
site_visit_analysis = site_visit_analysis_table(df_opw_cohort)
executive_summary_df = pd.concat([exec_summary, pd.DataFrame([{}]), exec_highlights], ignore_index=True)

OUTPUT_XLSX = Path("OPW_Episode_Cohort_Analysis1.xlsx")
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    _format_sheet(writer, "01_Full_Episode_Data", df_opw_cohort)
    _format_sheet(writer, "02_Cohort_Summary", cohort_summary_df)
    _format_sheet(writer, "03_Episode_Averages", episode_avgs)
    _format_sheet(writer, "04_Episode_Change", episode_change)
    _format_sheet(writer, "05_Episode_Sample_Size", episode_sample_size)
    for source, sheet in [("Orange", "06_Orange_Episode_Avg"), ("Purple", "07_Purple_Episode_Avg"), ("WhatsApp", "08_WhatsApp_Episode_Avg")]:
        if source in source_avgs:
            _format_sheet(writer, sheet, source_avgs[source])
    _format_sheet(writer, "09_Source_Comparison", source_comparison)
    _format_sheet(writer, "09b_Source_Overall_Avg", source_overall_avg)
    _format_sheet(writer, "10_Reactivation_Comparison", reactivation_comparison)
    _format_sheet(writer, "11_Participation", participation)
    _format_sheet(writer, "12_Conditional_Intensity", conditional_intensity)
    _format_sheet(writer, "13_Within_Lead_Pairs", within_lead_pairs)
    _format_sheet(writer, "14_Stage_Transitions", stage_transitions)
    _format_sheet(writer, "15_Site_Visit_Analysis", site_visit_analysis)
    _format_sheet(writer, "16_Executive_Summary", executive_summary_df)

print("Cohort filter logic: OR across Orange / Purple / WhatsApp qualifying events")
print("Orange/Purple: timestamp event in 6-month window (any original birth source)")
print("WhatsApp: born OR reactivated OR resubmitted in 6-month window (any original birth source for reactivation/resubmission)")
print(f"  Orange leads:   {(cohort_summary['is_orange_lead'] == 1).sum():,}")
print(f"  Purple leads:   {(cohort_summary['is_purple_lead'] == 1).sum():,}")
print(f"  WhatsApp leads: {(cohort_summary['is_whatsapp_lead'] == 1).sum():,}")
print(f"  Multi-channel:  {(cohort_summary[['is_orange_lead','is_purple_lead','is_whatsapp_lead']].sum(axis=1) > 1).sum():,}")
print(f"Qualified leads: {df_opw_cohort['lead'].nunique():,}")
print(f"Qualified episodes: {len(df_opw_cohort):,}")
print(f"Workbook written to: {OUTPUT_XLSX.resolve()}")



Cohort filter logic: OR across Orange / Purple / WhatsApp qualifying events
Orange/Purple: timestamp event in 6-month window (any original birth source)
WhatsApp: born OR reactivated OR resubmitted in 6-month window (any original birth source for reactivation/resubmission)
  Orange leads:   2,759
  Purple leads:   31,303
  WhatsApp leads: 34,481
  Multi-channel:  31,080
Qualified leads: 34,735
Qualified episodes: 45,115
Workbook written to: /Users/lekhanigam/RM Feature Analysis/OPW_Episode_Cohort_Analysis1.xlsx


In [25]:
# 01 — full episode dataset (preview; Excel has all rows)
full_data_preview_cols = [
    "lead", "episode_number", "episode_group", "episode_start_time", "episode_end_time",
    "project", "bornSource", "source", "episode_source", "start_stage", "end_stage",
    "total_web_timespent", "active_session_count", "episode_whatsapp_inbound_count",
    "total_answered_calls", "total_outbound_calls", "pickup_ratio",
    "resubmission_count_in_episode", "did_reach_site_visit_in_episode",
    "episode_stage_movement_count", "episode_duration_days",
    "is_orange_lead", "is_purple_lead", "is_whatsapp_lead",
]
print("=" * 110)
print("01_Full_Episode_Data")
print("=" * 110)
print(f"Rows: {len(df_opw_cohort):,}  ·  Columns in Excel: {df_opw_cohort.shape[1]}")
print(f"Notebook preview columns: {full_data_preview_cols}")
display_df(df_opw_cohort, keep_cols=full_data_preview_cols, rows=10)


01_Full_Episode_Data
Rows: 45,115  ·  Columns in Excel: 73
Notebook preview columns: ['lead', 'episode_number', 'episode_group', 'episode_start_time', 'episode_end_time', 'project', 'bornSource', 'source', 'episode_source', 'start_stage', 'end_stage', 'total_web_timespent', 'active_session_count', 'episode_whatsapp_inbound_count', 'total_answered_calls', 'total_outbound_calls', 'pickup_ratio', 'resubmission_count_in_episode', 'did_reach_site_visit_in_episode', 'episode_stage_movement_count', 'episode_duration_days', 'is_orange_lead', 'is_purple_lead', 'is_whatsapp_lead']


,lead,episode_number,episode_group,episode_start_time,episode_end_time,project,bornSource,source,episode_source,start_stage,...,total_answered_calls,total_outbound_calls,pickup_ratio,resubmission_count_in_episode,did_reach_site_visit_in_episode,episode_stage_movement_count,episode_duration_days,is_orange_lead,is_purple_lead,is_whatsapp_lead
0,6153206feb9f036ae4ee6853,1,Episode 1,2021-10-02 04:19:00.567,2023-07-07 03:29:09.776,SPECTRA,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,0,0.00,0,0,2,642.97,0,0,1
1,6154689550c5a7227fe16af9,1,Episode 1,2021-10-01 06:55:11.911,2023-07-07 03:29:18.199,SPRINGS,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,0,0.00,0,0,2,643.86,0,0,1
2,6154689550c5a7227fe16af9,2,Episode 2,2023-07-07 03:29:18.200,2026-04-10 07:20:33.512,SPRINGS,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,1,0.00,0,0,1,1008.16,0,0,1
3,615468b950c5a7227fe16d74,1,Episode 1,2021-10-01 06:55:14.716,2023-07-07 03:29:11.636,SPRINGS,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,0,0.00,0,0,2,643.86,0,0,1
4,615468b950c5a7227fe16d74,2,Episode 2,2023-07-07 03:29:11.637,2026-02-20 10:35:46.634,SPRINGS,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,1,0.00,0,0,2,959.30,0,0,1
5,615468b950c5a7227fe16d74,3,Episode 3,2026-02-20 10:35:46.635,2026-04-04 08:12:48.110,SPRINGS,Channel Partner,ASBL DB,Channel Partner,New Lead,...,0,3,0.00,0,0,3,42.90,0,0,1
6,616340b0dc442c4c16401cb5,1,Episode 1,2021-10-10 19:36:16.347,2023-07-07 03:28:47.984,SPECTRA,Web Lead,Web Lead,Web Lead,Lead Initiated,...,2,2,0.50,0,0,2,634.33,0,0,1
7,616340b0dc442c4c16401cb5,2,Episode 2,2023-07-07 03:28:47.985,2025-05-22 10:46:37.282,SPECTRA,Web Lead,Web Lead,Web Lead,Lead Initiated,...,0,8,0.00,0,0,2,685.30,0,0,1
8,616340b0dc442c4c16401cb5,3,Episode 3,2025-05-22 10:46:37.283,2026-05-14 09:47:16.313,SPECTRA,Web Lead,Web Lead,Web Lead,Lead Initiated,...,1,13,0.17,0,0,2,356.96,0,0,1
9,6166669236e6276bd6f6d0e3,1,Episode 1,2021-10-13 04:54:42.623,2023-07-07 03:28:47.477,SPECTRA,Web Lead,Web Lead,Web Lead,Lead Initiated,...,1,1,1.00,0,0,2,631.94,0,0,1


In [26]:
# 02 + 05 — cohort counts and episode sample sizes
show_sheet("02_Cohort_Summary", cohort_summary_df)
show_sheet("05_Episode_Sample_Size", episode_sample_size)


02_Cohort_Summary
              metric  value
  total_unique_leads  34735
      total_episodes  45115
     episode_1_count  34735
     episode_2_count   7608
     episode_3_count   1676
     episode_4_count    505
episode_5_plus_count    591
        orange_leads   2759
        purple_leads  31303
      whatsapp_leads  34481
05_Episode_Sample_Size
   Episode  Unique Leads  Total Episodes
 Episode 1         34735           34735
 Episode 2          7608            7608
 Episode 3          1676            1676
 Episode 4           505             505
Episode 5+           214             591


,Episode,Unique Leads,Total Episodes
0,Episode 1,34735,34735
1,Episode 2,7608,7608
2,Episode 3,1676,1676
3,Episode 4,505,505
4,Episode 5+,214,591


In [27]:
# 03 + 04 — View A episode averages and episode-to-episode change
print("Averages exclude zeros and IQR outliers; 0/1 flags stay as rates.")
show_sheet("03_Episode_Averages", episode_avgs)
show_sheet("04_Episode_Change", episode_change)


Averages exclude zeros and IQR outliers; 0/1 flags stay as rates.
03_Episode_Averages
                        Feature  Episode 1  Episode 2  Episode 3  Episode 4  Episode 5+
          total_stage_movements       8.67       7.24       7.28       7.44        6.11
         is_reactivated_episode       0.22       0.22       0.30       0.42        0.64
            total_web_timespent      53.34     124.45     401.71     434.04     3081.09
           active_session_count       1.34       1.72       2.27       2.39       14.28
                 home_timespent      34.17      73.97     171.39     190.54      997.22
                price_timespent      69.00     166.96     203.80     183.02      708.87
                 plan_timespent      63.61     168.10     276.04     356.23      964.52
             location_timespent      34.69      76.35     161.75     193.08      428.74
            amenities_timespent      22.01      98.84     264.16     281.63      418.52
                media_timespent   

,Feature,Overall Trend,Episode 1 Avg,Episode 2 Avg,Episode 3 Avg,Episode 4 Avg,Episode 5+ Avg,Δ 2→1,% 2→1,Δ 3→2,% 3→2,Δ 4→3,% 4→3,Δ 5+→4,% 5+→4
0,total_stage_movements,Mixed,8.67,7.24,7.28,7.44,6.11,-1.42,-16.40%,0.04,0.50%,0.16,2.21%,-1.33,-17.87%
1,is_reactivated_episode,Increasing,0.22,0.22,0.30,0.42,0.64,0.00,0.58%,0.08,36.78%,0.12,40.64%,0.21,50.53%
2,total_web_timespent,Increasing,53.34,124.45,401.71,434.04,3081.09,71.11,133.33%,277.26,222.79%,32.32,8.05%,2647.05,609.87%
3,active_session_count,Increasing,1.34,1.72,2.27,2.39,14.28,0.39,28.89%,0.55,31.75%,0.12,5.51%,11.89,496.78%
4,home_timespent,Increasing,34.17,73.97,171.39,190.54,997.22,39.80,116.47%,97.41,131.69%,19.16,11.18%,806.68,423.36%
5,price_timespent,Mixed,69.00,166.96,203.80,183.02,708.87,97.95,141.95%,36.84,22.07%,-20.78,-10.19%,525.86,287.32%
6,plan_timespent,Increasing,63.61,168.10,276.04,356.23,964.52,104.49,164.27%,107.94,64.21%,80.19,29.05%,608.29,170.76%
7,location_timespent,Increasing,34.69,76.35,161.75,193.08,428.74,41.66,120.08%,85.40,111.86%,31.34,19.37%,235.65,122.05%
8,amenities_timespent,Increasing,22.01,98.84,264.16,281.63,418.52,76.83,348.99%,165.32,167.26%,17.47,6.61%,136.89,48.61%
9,media_timespent,Mixed,102.17,86.46,349.74,797.74,888.01,-15.71,-15.38%,263.28,304.52%,448.00,128.09%,90.27,11.32%


In [28]:
# 06–09 — Orange / Purple / WhatsApp source slices
for source, sheet in [("Orange", "06_Orange_Episode_Avg"), ("Purple", "07_Purple_Episode_Avg"), ("WhatsApp", "08_WhatsApp_Episode_Avg")]:
    if source in source_avgs:
        show_sheet(sheet, source_avgs[source])
show_sheet("09_Source_Comparison", source_comparison)
show_sheet("09b_Source_Overall_Avg", source_overall_avg, drop_cols=["Aggregation Level"])


06_Orange_Episode_Avg
                        Feature  Episode 1  Episode 2  Episode 3  Episode 4  Episode 5+
          total_stage_movements      11.73       8.33       8.01       8.50        8.40
         is_reactivated_episode       0.34       0.31       0.41       0.46        0.66
            total_web_timespent     112.50     164.78     182.31     630.71     2290.87
           active_session_count       1.37       1.38       2.35       2.00       13.19
                 home_timespent      62.30      77.11     111.57     215.62     1156.87
                price_timespent      93.15     167.80     183.52     170.34     1513.71
                 plan_timespent     115.60     132.14     173.24     169.19     1658.24
             location_timespent      23.17     181.32      15.77     172.74      373.39
            amenities_timespent      24.91      51.87       4.23     100.70      284.14
                media_timespent     129.33    1173.88     404.88     105.17      937.11
       spe

,Feature,Orange Overall Avg,Purple Overall Avg,WhatsApp Overall Avg
0,total_stage_movements,10.55,8.35,8.24
1,is_reactivated_episode,0.35,0.23,0.23
2,total_web_timespent,183.14,101.27,111.07
3,active_session_count,1.81,1.67,1.66
4,home_timespent,95.93,61.08,63.59
5,price_timespent,182.12,205.60,192.16
6,plan_timespent,194.68,158.98,164.82
7,location_timespent,109.08,85.90,89.83
8,amenities_timespent,82.74,120.87,117.20
9,media_timespent,699.25,479.38,444.44


In [29]:
# 10–12 — reactivation, participation, conditional intensity
show_sheet("10_Reactivation_Comparison", reactivation_comparison)
show_sheet("11_Participation", participation)
show_sheet("12_Conditional_Intensity", conditional_intensity)


10_Reactivation_Comparison
                        Feature  Non-Reactivated Avg  Reactivated Avg  Absolute Change     % Change
          total_stage_movements                 8.35             7.86            -0.49       -5.84%
         is_reactivated_episode                 0.00             1.00             1.00 New Activity
            total_web_timespent                66.04           137.03            70.99      107.50%
           active_session_count                 1.36             1.71             0.35       26.04%
                 home_timespent                41.81            78.59            36.78       87.97%
                price_timespent               101.23           218.04           116.81      115.38%
                 plan_timespent               107.10           227.48           120.38      112.40%
             location_timespent                70.29           101.37            31.09       44.23%
            amenities_timespent                56.79           147.22    

,Feature,Cohort,Active Episodes,Mean,Median,P75,P90,P95
0,total_stage_movements,Non-Reactivated,33252,8.35,7.00,12.00,16.00,19.00
1,total_stage_movements,Reactivated,9876,7.86,6.00,11.00,15.00,18.00
2,is_reactivated_episode,Non-Reactivated,0,NaN,NaN,NaN,NaN,NaN
3,is_reactivated_episode,Reactivated,10380,1.00,1.00,1.00,1.00,1.00
4,total_web_timespent,Non-Reactivated,539,66.04,26.16,89.18,206.87,272.77
...,...,...,...,...,...,...,...,...
65,total_call_duration,Reactivated,5785,50.57,33.00,61.87,118.91,160.11
66,pickup_ratio,Non-Reactivated,24930,0.69,0.80,1.00,1.00,1.00
67,pickup_ratio,Reactivated,6763,0.72,1.00,1.00,1.00,1.00
68,avg_web_activity,Non-Reactivated,557,44.63,19.04,62.79,131.25,164.51


In [30]:
# 13 — View B within-lead pairs
show_sheet("13_Within_Lead_Pairs", within_lead_pairs, drop_cols=["test_used", "p_value", "% unchanged"])


13_Within_Lead_Pairs
    pair                         feature  active_pair_count  previous_average  current_average  mean_shift   % change  % increased  % decreased significance
  2 -> 1           total_stage_movements               7608              8.36             7.24       -1.12    -13.35%        43.44        48.71       p<0.05
  3 -> 2           total_stage_movements               1676              6.82             7.28        0.46      6.73%        48.21        42.60           NS
  4 -> 3           total_stage_movements                505              7.29             7.44        0.15      2.08%        49.50        41.39       p<0.05
5+ -> 4+           total_stage_movements                591              5.41             6.11        0.71     13.05%        45.18        39.76           NS
  2 -> 1          is_reactivated_episode               7608              1.00             0.22       -0.78    -77.97%         0.00        77.97       p<0.05
  3 -> 2          is_reactivated_epis

,pair,feature,active_pair_count,previous_average,current_average,mean_shift,% change,% increased,% decreased,significance
0,2 -> 1,total_stage_movements,7608,8.36,7.24,-1.12,-13.35%,43.44,48.71,p<0.05
1,3 -> 2,total_stage_movements,1676,6.82,7.28,0.46,6.73%,48.21,42.60,NS
2,4 -> 3,total_stage_movements,505,7.29,7.44,0.15,2.08%,49.50,41.39,p<0.05
3,5+ -> 4+,total_stage_movements,591,5.41,6.11,0.71,13.05%,45.18,39.76,NS
4,2 -> 1,is_reactivated_episode,7608,1.00,0.22,-0.78,-77.97%,0.00,77.97,p<0.05
...,...,...,...,...,...,...,...,...,...,...
135,5+ -> 4+,pickup_ratio,591,0.76,0.72,-0.03,-4.15%,24.70,26.40,NS
136,2 -> 1,avg_web_activity,7608,40.19,81.35,41.16,102.41%,5.24,8.27,NS
137,3 -> 2,avg_web_activity,1676,115.32,121.89,6.57,5.70%,5.49,11.69,NS
138,4 -> 3,avg_web_activity,505,135.40,139.96,4.56,3.37%,9.90,12.28,NS


In [32]:
# 14–16 — stage transitions, site visit, executive summary
show_sheet("14_Site_Visit_Analysis", site_visit_analysis)



14_Site_Visit_Analysis
   Episode  Episodes  did_reach_site_visit_in_episode avg  did_reach_site_visit_in_episode % active  is_site_visit_book_clicked avg  is_site_visit_book_clicked % active
 Episode 1     34735                                 0.01                                      1.28                            0.13                                 0.48
 Episode 2      7608                                 0.02                                      2.04                            0.08                                 0.66
 Episode 3      1676                                 0.03                                      3.28                            0.12                                 1.01
 Episode 4       505                                 0.04                                      4.36                            0.08                                 1.19
Episode 5+       591                                 0.05                                      5.25                            0.26 

,Episode,Episodes,did_reach_site_visit_in_episode avg,did_reach_site_visit_in_episode % active,is_site_visit_book_clicked avg,is_site_visit_book_clicked % active
0,Episode 1,34735,0.01,1.28,0.13,0.48
1,Episode 2,7608,0.02,2.04,0.08,0.66
2,Episode 3,1676,0.03,3.28,0.12,1.01
3,Episode 4,505,0.04,4.36,0.08,1.19
4,Episode 5+,591,0.05,5.25,0.26,10.49
